In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('../../../files/movie_list.csv')

data.columns = ['Title','Director','Screenwriter','Synopsis','Score','NumOfScore','Actors','ScoresOfComment','Comment','Genre','Country','Viewer','RunningTime','Adult']
data = data.dropna(axis=0).reset_index(drop=True)

data.head()

ModuleNotFoundError: No module named 'pandas'

유저 taste elicit에서 아래의 사항들의 분류의 기준으로 적용\
- KOFIC 보고서에서 발췌한 영화 선택 요인들 (중요도 내림차순)
  - 줄거리, 소재
  - 장르
  - 출연배우
  - 다른 관객 평점, 리뷰, 입소문
  - 예고편, 포스터 등의 광고 및 홍보
  - 예매 순위 및 흥행
  - 함께 보는 사람의 취향
  - 영상미, 시각효과
  - 시리즈물 여부
  - 원작 화제성
  - 음악, 음향효과
  - 감독
  - 러닝타임
  - 제작 국가
  - 영화제 수상, 전문가 평가
  - 상영등급
  - 제작비 규모
  
- 사용자의 취향
  - 줄거리, 소재
  - 장르
  - 출연배우
  - 영상미, 시각효과
  - 시리즈물 여부
  - 원작 화제성
  - 음악, 음향효과
  - 감독
  - 러닝타임
  - 제작 국가


: 완성도 <-> 줄거리 내용 이건 분명히 계층이 다른 분류 같은데...

In [2]:
! conda install langchain_openai

zsh:1: command not found: conda


In [3]:
import openai

from langchain_openai import ChatOpenAI
from langchain import LLMChain, PromptTemplate


## Load API
# load_dotenv()
# api_key = os.getenv('OPENAI_API_KEY')

llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0)

## 사용자로부터 취향 추출하기  (민규님 코드 발췌)
> Q: 기억나는 영화와 그 영화의 좋은 점과 싫은 점을 이야기해줘

### Version 1

KOFIC 만 참고해서 아래를 분류 기준으로 두기

Plot, Theme, Genre, Cast, Visual Effects, Series Status, Source Material, OST, Sound Effects, Director, Runtime, Country of Production

In [4]:
import json

def parse_pref_to_json(pref):
    return json.loads(pref.replace("json", "").strip().replace("```", "")) 

In [5]:

prompt = """You are a movie preference evaluator. The user will talk about what they liked and disliked about a specific movie. Based on this, evaluate the user's preferences by extracting the reasons for liking and disliking the movie by feature, and return the result in the following format:

The JSON object should include the following fields:
movie_title: The movie title based on the user's input.
features: A list of features where each feature contains.
  - feature_name: The name of the feature
  - likes: The reasons the user liked the movie, categorized by feature.
  - dislikes: The reasons the user disliked the movie, categorized by feature.

feature_name should be one of Plot, Theme, Genre, Cast, Visual Effects, Series Status, Source Material, OST, Sound Effects, Director, Runtime, Country of Production

User input: {query}
JSON object:
"""

prompt_template = PromptTemplate(template=prompt, input_variables=["query"])
preference_eliction_chain = prompt_template | llm

result_json1 = preference_eliction_chain.invoke({"query":"어바웃타임에서 아버지와 아들이 시간에 대해서 이야기하는 장면과 나중에 아들(주인공)이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 너무 인상깊었고, 또 로맨스적인 요소도 너무 좋았어"}).content
result_json2 = preference_eliction_chain.invoke({"query":"베테랑2에서 좋았던 점은 정해인 배우의 액션이 인상 깊었어. 굉장히 빠르게 진행되어서 좋았고 타격이 아닌 그라운드 기술이 중점이 된 액션이라 신선했어. 다만 스토리가 너무 허술해서 아쉬웠고, 특히 마지막 엔딩은 갑자기 끝난 느낌? 결말이 전혀 매력적으로 느껴지지 않았어"}).content
result_json3 = preference_eliction_chain.invoke({"query":"아메리칸 셰프 영화를 봤는데, 신나는 음악과 신나는 요리 편집이 보기에 되게 시원했어. 그리고 주인공이 아들과의 관계를 회복하는 과정도 되게 재밌었어."}).content
result_json4 = preference_eliction_chain.invoke({"query":"검사외전을 봤는데 너무 별로였어. 강동원 특유의 신나고 재밌는 분위기는 좋았지만 스토리가 너무 별로였어. 문제가 해결되는 과정이 너무 얼렁뚱땅 넘어가는 느낌이라고 할까? 전혀 설득력이 없었어."}).content
result_json1 = parse_pref_to_json(result_json1)
result_json2 = parse_pref_to_json(result_json2)
result_json3 = parse_pref_to_json(result_json3)
result_json4 = parse_pref_to_json(result_json4)
print(result_json1)
print(result_json2)
print(result_json3)
print(result_json4)

{'movie_title': '어바웃타임', 'features': [{'feature_name': 'Plot', 'likes': ['아버지와 아들이 시간에 대해서 이야기하는 장면이 인상적이었다.', '주인공이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 좋았다.'], 'dislikes': []}, {'feature_name': 'Theme', 'likes': ['로맨스적인 요소가 좋았다.'], 'dislikes': []}, {'feature_name': 'Genre', 'likes': [], 'dislikes': []}, {'feature_name': 'Cast', 'likes': [], 'dislikes': []}, {'feature_name': 'Visual Effects', 'likes': [], 'dislikes': []}, {'feature_name': 'Series Status', 'likes': [], 'dislikes': []}, {'feature_name': 'Source Material', 'likes': [], 'dislikes': []}, {'feature_name': 'OST', 'likes': [], 'dislikes': []}, {'feature_name': 'Sound Effects', 'likes': [], 'dislikes': []}, {'feature_name': 'Director', 'likes': [], 'dislikes': []}, {'feature_name': 'Runtime', 'likes': [], 'dislikes': []}, {'feature_name': 'Country of Production', 'likes': [], 'dislikes': []}]}
{'movie_title': '베테랑2', 'features': [{'feature_name': 'Cast', 'likes': ['정해인 배우의 액션이 인상 깊었다.'], 'dislikes': []}, {'feature_name': 'Visual Eff

In [6]:
result_json1 = preference_eliction_chain.invoke({"query":"어바웃타임에서 아버지와 아들이 시간에 대해서 이야기하는 장면과 나중에 아들(주인공)이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 너무 인상깊었고, 또 로맨스적인 요소도 너무 좋았어"}).content
result_json2 = preference_eliction_chain.invoke({"query":"베테랑2에서 좋았던 점은 정해인 배우의 액션이 인상 깊었어. 굉장히 빠르게 진행되어서 좋았고 타격이 아닌 그라운드 기술이 중점이 된 액션이라 신선했어. 다만 스토리가 너무 허술해서 아쉬웠고, 특히 마지막 엔딩은 갑자기 끝난 느낌? 결말이 전혀 매력적으로 느껴지지 않았어"}).content
result_json3 = preference_eliction_chain.invoke({"query":"아메리칸 셰프 영화를 봤는데, 신나는 음악과 신나는 요리 편집이 보기에 되게 시원했어. 그리고 주인공이 아들과의 관계를 회복하는 과정도 되게 재밌었어."}).content
result_json4 = preference_eliction_chain.invoke({"query":"검사외전을 봤는데 너무 별로였어. 강동원 특유의 신나고 재밌는 분위기는 좋았지만 스토리가 너무 별로였어. 문제가 해결되는 과정이 너무 얼렁뚱땅 넘어가는 느낌이라고 할까? 전혀 설득력이 없었어."}).content

result_json1 = parse_pref_to_json(result_json1)
result_json2 = parse_pref_to_json(result_json2)
result_json3 = parse_pref_to_json(result_json3)
result_json4 = parse_pref_to_json(result_json4)

print(result_json1)
print(result_json2)
print(result_json3)
print(result_json4)

{'movie_title': '어바웃타임', 'features': [{'feature_name': 'Plot', 'likes': ['아버지와 아들이 시간에 대해서 이야기하는 장면이 인상적이었다.', '주인공이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 좋았다.'], 'dislikes': []}, {'feature_name': 'Theme', 'likes': ['로맨스적인 요소가 좋았다.'], 'dislikes': []}, {'feature_name': 'Genre', 'likes': [], 'dislikes': []}, {'feature_name': 'Cast', 'likes': [], 'dislikes': []}, {'feature_name': 'Visual Effects', 'likes': [], 'dislikes': []}, {'feature_name': 'Series Status', 'likes': [], 'dislikes': []}, {'feature_name': 'Source Material', 'likes': [], 'dislikes': []}, {'feature_name': 'OST', 'likes': [], 'dislikes': []}, {'feature_name': 'Sound Effects', 'likes': [], 'dislikes': []}, {'feature_name': 'Director', 'likes': [], 'dislikes': []}, {'feature_name': 'Runtime', 'likes': [], 'dislikes': []}, {'feature_name': 'Country of Production', 'likes': [], 'dislikes': []}]}
{'movie_title': '베테랑2', 'features': [{'feature_name': 'Cast', 'likes': ['정해인 배우의 액션이 인상 깊었다.'], 'dislikes': []}, {'feature_name': 'Visual Eff

In [7]:
dict_list = [result_json1, result_json2, result_json3, result_json4]

combined_dict = {}

for movie_dict in dict_list:
    
    movie_title = movie_dict['movie_title']
    combined_dict[movie_title] = []
    
    for feat in movie_dict['features']:

        combined_dict[movie_title].append({
            'feature_name': feat['feature_name'],
            'likes': feat['likes'],
            'dislikes': feat['dislikes']
        })

combined_json = json.dumps(combined_dict, ensure_ascii=False,  indent=4)

In [8]:
print(combined_dict['베테랑2'])

[{'feature_name': 'Cast', 'likes': ['정해인 배우의 액션이 인상 깊었다.'], 'dislikes': []}, {'feature_name': 'Visual Effects', 'likes': ['굉장히 빠르게 진행되어서 좋았다.', '타격이 아닌 그라운드 기술이 중점이 된 액션이라 신선했다.'], 'dislikes': []}, {'feature_name': 'Plot', 'likes': [], 'dislikes': ['스토리가 너무 허술했다.', '특히 마지막 엔딩은 갑자기 끝난 느낌이었다.', '결말이 전혀 매력적으로 느껴지지 않았다.']}]


뭔가 여전히 likes의 분류가 plot인 거는 OK\
그런데, 취향과 관련하려면 **왜, 뭐가 좋았는 지** 필요함.

### Version 2.

+ 감정포인트 추가
  
CGV것도 참고함

1. 매력 포인트\
스토리, 감독연출, OST, 배우 연기, 영상미
1. 감정 포인트 **줄거리(Plot)에 집어넣음** \
즐거움, 스트레스 해소, 감동, 몰입감, 긴장감


In [9]:

prompt = """You are a movie preference evaluator. The user will talk about what they liked and disliked about a specific movie. Based on this, evaluate the user's preferences by extracting the reasons for liking and disliking the movie by feature, and return the result in the following format:

The JSON object should include the following fields:
movie_title: The movie title based on the user's input.
features: A list of features where each feature contains.
  - feature_name: The name of the feature
  - likes: The reasons the user liked the movie, categorized by feature.
  - dislikes: The reasons the user disliked the movie, categorized by feature.
Emotion Category: should be one of [즐거움, 스트레스 해소, 감동, 몰입감, 긴장감]

feature_name should be one of Plot, Theme, Genre, Cast, Visual Effects, Series Status, Source Material, OST, Sound Effects, Director, Runtime, Country of Production

User input: {query}
JSON object:
"""

prompt_template = PromptTemplate(template=prompt, input_variables=["query"])
preference_eliction_chain = prompt_template | llm

result_json1 = preference_eliction_chain.invoke({"query":"어바웃타임에서 아버지와 아들이 시간에 대해서 이야기하는 장면과 나중에 아들(주인공)이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 너무 인상깊었고, 또 로맨스적인 요소도 너무 좋았어"}).content
result_json2 = preference_eliction_chain.invoke({"query":"베테랑2에서 좋았던 점은 정해인 배우의 액션이 인상 깊었어. 굉장히 빠르게 진행되어서 좋았고 타격이 아닌 그라운드 기술이 중점이 된 액션이라 신선했어. 다만 스토리가 너무 허술해서 아쉬웠고, 특히 마지막 엔딩은 갑자기 끝난 느낌? 결말이 전혀 매력적으로 느껴지지 않았어"}).content
result_json3 = preference_eliction_chain.invoke({"query":"아메리칸 셰프 영화를 봤는데, 신나는 음악과 신나는 요리 편집이 보기에 되게 시원했어. 그리고 주인공이 아들과의 관계를 회복하는 과정도 되게 재밌었어."}).content
result_json4 = preference_eliction_chain.invoke({"query":"검사외전을 봤는데 너무 별로였어. 강동원 특유의 신나고 재밌는 분위기는 좋았지만 스토리가 너무 별로였어. 문제가 해결되는 과정이 너무 얼렁뚱땅 넘어가는 느낌이라고 할까? 전혀 설득력이 없었어."}).content

result_json1 = parse_pref_to_json(result_json1)
result_json2 = parse_pref_to_json(result_json2)
result_json3 = parse_pref_to_json(result_json3)
result_json4 = parse_pref_to_json(result_json4)

print(result_json1)
print(result_json2)
print(result_json3)
print(result_json4)

{'movie_title': '어바웃타임', 'features': [{'feature_name': 'Plot', 'likes': ['아버지와 아들이 시간에 대해서 이야기하는 장면이 인상 깊었다.', '주인공이 더 이상 시간 여행을 하지 않고 현재에 살아간다는 점이 좋았다.'], 'dislikes': []}, {'feature_name': 'Theme', 'likes': ['로맨스적인 요소가 좋았다.'], 'dislikes': []}], 'Emotion Category': '감동'}
{'movie_title': '베테랑2', 'features': [{'feature_name': 'Cast', 'likes': ['정해인 배우의 액션이 인상 깊었다.'], 'dislikes': []}, {'feature_name': 'Plot', 'likes': ['굉장히 빠르게 진행되어서 좋았다.'], 'dislikes': ['스토리가 너무 허술했다.', '마지막 엔딩이 갑자기 끝난 느낌이었다.', '결말이 전혀 매력적으로 느껴지지 않았다.']}, {'feature_name': 'Genre', 'likes': ['타격이 아닌 그라운드 기술이 중점이 된 액션이라 신선했다.'], 'dislikes': []}], 'Emotion Category': '몰입감'}
{'movie_title': '아메리칸 셰프', 'features': [{'feature_name': 'OST', 'likes': ['신나는 음악이 좋았다.'], 'dislikes': []}, {'feature_name': 'Visual Effects', 'likes': ['신나는 요리 편집이 시원했다.'], 'dislikes': []}, {'feature_name': 'Plot', 'likes': ['주인공이 아들과의 관계를 회복하는 과정이 재밌었다.'], 'dislikes': []}], 'Emotion Category': '즐거움'}
{'movie_title': '검사외전', 'features': [{'feature_name'

In [10]:
dict_list = [result_json1, result_json2, result_json3, result_json4]

combined_dict = {}

for movie_dict in dict_list:
    
    movie_title = movie_dict['movie_title']
    combined_dict[movie_title] = []
    
    for feat in movie_dict['features']:

        combined_dict[movie_title].append({
            'feature_name': feat['feature_name'],
            'likes': feat['likes'],
            'dislikes': feat['dislikes']
        })

combined_json = json.dumps(combined_dict, ensure_ascii=False,  indent=4)

In [24]:
print(result_json4)

{'movie_title': '검사외전', 'features': [{'feature_name': 'Plot', 'likes': [], 'dislikes': ['스토리가 너무 별로였어.', '문제가 해결되는 과정이 너무 얼렁뚱땅 넘어가는 느낌이라고 할까?', '전혀 설득력이 없었어.']}, {'feature_name': 'Cast', 'likes': ['강동원 특유의 신나고 재밌는 분위기는 좋았지만.'], 'dislikes': []}], 'Emotion Category': '스트레스 해소'}


# 영화 리뷰도 같은 방식으로 만들기

In [23]:
import pandas as pd
import ast

df = pd.read_csv('../../../files/preprocessed_movie_sample.csv', converters={'director': ast.literal_eval, 'screenwriter': ast.literal_eval, 'actors': ast.literal_eval, 'comment_ratings': ast.literal_eval, 'comment_text': ast.literal_eval})
df.head()

,title,director,screenwriter,plot,rating,rating_count,actors,comment_ratings,comment_texts,genres,countries,running_time,adult
0,월플라워,[스티븐 크보스키],[스티븐 크보스키],말 못할 트라우마를 가지고 자신만의 세계에 갇혀있던 ‘찰리’는 고등학교 신입생이 돼...,3.8,305000.0,[에즈라 밀러로건 메먼엠마 왓슨],[4.05.04.02.04.04.03.54.53.54.5],['그 터널을 무사히 빠져나온 모두에게사람은 자신의 크기에 맞는 사랑을 한다샘이라는...,"['드라마', '로맨스']",['미국'],102.0,0.0
1,기생충,[봉준호],[봉준호한진원],“폐 끼치고 싶진 않았어요” 전원백수로 살 길 막막하지만 사이는 좋은 기택(송강호)...,4.3,1265000.0,[박소담송강호조여정최우식이선균장혜진이정은],[5.04.55.00.54.55.04.05.05.05.0],['황금종려상 받은 영화를 자막없이 볼 수 있는 행복 190529상승과 하강으로 명...,['드라마'],['한국'],131.0,0.0
2,메멘토,[크리스토퍼 놀란],[크리스토퍼 놀란],"아내가 살해당한 후, 10분밖에 기억 못하는 단기기억상실증에 걸린 남자가 사진, 메...",4.1,641000.0,[조 판톨리아노가이 피어스캐리 앤 모스],[5.05.04.55.05.05.05.04.05.04.0],['사람은 보고싶은 것만 보고 듣고싶은 것만 듣고 믿고 싶은 것만 믿고 기억하고 싶...,"['미스터리', '스릴러']",['미국'],113.0,0.0
3,노인을 위한 나라는 없다,[에단 코엔조엘 코엔],[에단 코엔조엘 코엔],르웰린 모스(조쉬 브롤린)는 총격전이 벌어진 끔찍한 현장에서 우연히 이백만 달러가 ...,4.0,349000.0,[토미 리 존스하비에르 바르뎀조쉬 브롤린],[5.05.05.05.03.55.05.04.55.05.0],['그거 아세요? 이 영화에는 배경음악이 단 일초도 쓰이지 않았다는것피튀기는 젊은 ...,"['범죄', '드라마', '스릴러']",['미국'],122.0,1.0
4,파이트 클럽,[데이비드 핀처],[짐 유힐],당신이 알고 있는 모든 것은 허구다! 비싼 가구들로 집 안을 채우지만 삶에 강한 공...,4.1,371000.0,[브래드 피트에드워드 노튼헬레나 본햄 카터],[5.05.05.04.54.05.04.05.03.04.5],['현대문명의 허상을 조롱하는 통렬한 블랙코미디.영화 자체가 섹시하다아주 많은 경우...,"['드라마', '액션']",['미국독일이탈리아'],139.0,1.0


In [21]:
result_json_월플라워 = preference_eliction_chain.invoke(df.loc[0, 'comment_texts']).content
result_json_월플라워 = parse_pref_to_json(result_json_월플라워)

In [22]:
result_json_월플라워

{'movie_title': '그 터널을 무사히 빠져나온 모두에게',
 'features': [{'feature_name': 'Plot',
   'likes': ['사랑과 우정의 관계를 통해 성장하는 이야기', '청춘의 아픔과 극복을 다룬 점'],
   'dislikes': ['자신의 과거와 비교하게 만들어 불쾌함을 느꼈음', '질투와 초라함을 느끼게 하는 내용']},
  {'feature_name': 'Theme',
   'likes': ['사랑과 우정의 복잡한 관계를 탐구', '청춘의 성장과 극복을 강조'],
   'dislikes': ['한국 청소년들의 상처를 다루는 점이 불편함', '자신의 상처를 드러내는 것이 힘들게 느껴짐']},
  {'feature_name': 'Cast',
   'likes': ['캐릭터들이 현실적이고 relatable함'],
   'dislikes': []},
  {'feature_name': 'Visual Effects', 'likes': [], 'dislikes': []},
  {'feature_name': 'OST', 'likes': [], 'dislikes': []},
  {'feature_name': 'Sound Effects', 'likes': [], 'dislikes': []},
  {'feature_name': 'Director', 'likes': [], 'dislikes': []},
  {'feature_name': 'Runtime', 'likes': [], 'dislikes': []},
  {'feature_name': 'Country of Production', 'likes': [], 'dislikes': []}],
 'Emotion': '감동'}